# Interactive Gillespie Module 2: SinI-SinR Stochastic Switch

This module is inspired by Lord et al., *Science* 2019. It simulates stochastic competition between two proteins:

- SinI and SinR are produced constitutively.
- SinI + SinR form an inert complex.
- Free SinR represses reporter Z expression.
- Reporter Z turns ON when free SinR is absent.

This is a simplified teaching model, not a full reproduction of the paper.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams.update({
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [2]:
def gillespie_sini_sinr(
    lambda_I=0.8, lambda_R=1.0, complex_rate=40.0, lifetime=1.0,
    reporter_prod=40.0, reporter_decay=0.5, tmax=100.0, seed=None
):
    """Simplified SSA for stochastic SinI-SinR antagonism.

    State vector: [I_free, R_free, C_complex, Z_reporter]
    Reactions:
      1. ∅ -> I                 rate lambda_I
      2. ∅ -> R                 rate lambda_R
      3. I + R -> C             rate complex_rate * I * R
      4. I -> ∅                 rate I / lifetime
      5. R -> ∅                 rate R / lifetime
      6. C -> ∅                 rate C / lifetime
      7. ∅ -> Z if R_free == 0  rate reporter_prod
      8. Z -> ∅                 rate reporter_decay * Z
    """
    rng = np.random.default_rng(seed)
    I, R, C, Z = 0, 0, 0, 0
    t = 0.0
    T, X = [t], [[I, R, C, Z]]

    while t < tmax:
        props = np.array([
            lambda_I,
            lambda_R,
            complex_rate * I * R,
            I / lifetime,
            R / lifetime,
            C / lifetime,
            reporter_prod if R == 0 else 0.0,
            reporter_decay * Z,
        ], dtype=float)
        a0 = props.sum()
        if a0 <= 0:
            break

        r1, r2 = rng.random(2)
        tau = -np.log(r1) / a0
        if t + tau > tmax:
            break
        t += tau
        mu = np.searchsorted(np.cumsum(props), r2 * a0)

        if mu == 0:
            I += 1
        elif mu == 1:
            R += 1
        elif mu == 2 and I > 0 and R > 0:
            I -= 1; R -= 1; C += 1
        elif mu == 3 and I > 0:
            I -= 1
        elif mu == 4 and R > 0:
            R -= 1
        elif mu == 5 and C > 0:
            C -= 1
        elif mu == 6:
            Z += 1
        elif mu == 7 and Z > 0:
            Z -= 1

        T.append(t)
        X.append([I, R, C, Z])

    return np.array(T), np.array(X)

In [3]:
def plot_switch(
    ratio_I_to_R=0.8, lambda_R=1.0, complex_rate=40.0, lifetime=1.0,
    reporter_prod=40.0, reporter_decay=0.5, tmax=100, seed=1
):
    lambda_I = ratio_I_to_R * lambda_R
    T, X = gillespie_sini_sinr(lambda_I, lambda_R, complex_rate, lifetime, reporter_prod, reporter_decay, tmax, seed)
    I, R, C, Z = X.T

    fig, ax1 = plt.subplots(figsize=(9, 4))
    ax1.step(T, I, where='post', lw=1.6, label='free SinI')
    ax1.step(T, R, where='post', lw=1.6, label='free SinR')
    ax1.set_xlabel('time')
    ax1.set_ylabel('free protein count')
    ax2 = ax1.twinx()
    ax2.step(T, Z, where='post', lw=1.3, alpha=0.8, label='reporter Z', color = 'grey')
    ax2.set_ylabel('reporter count')

    dt = np.diff(np.r_[T, tmax])
    frac_on = np.sum(dt * (Z > 10)) / np.sum(dt) if len(dt) else 0
    ax1.set_title(f'Stochastic switch trace | fraction reporter ON ≈ {frac_on:.2f}')
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1+h2, l1+l2, frameon=False, ncol=3, loc='upper left')
    plt.show()

interact(
    plot_switch,
    ratio_I_to_R=FloatSlider(value=0.8, min=0.1, max=2, step=0.02, description='λI/λR'),
    lambda_R=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='λR'),
    complex_rate=FloatSlider(value=40.0, min=0.1, max=100.0, step=1.0, description='complex'),
    lifetime=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='lifetime'),
    reporter_prod=FloatSlider(value=40.0, min=1, max=120, step=1, description='Z prod'),
    reporter_decay=FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05, description='Z decay'),
    tmax=IntSlider(value=100, min=20, max=300, step=10, description='Tmax'),
    seed=IntSlider(value=1, min=0, max=1000, step=1, description='seed'),
);

interactive(children=(FloatSlider(value=0.8, description='λI/λR', max=2.0, min=0.1, step=0.02), FloatSlider(va…

In [4]:
def sweep_ratio(lambda_R=1.0, complex_rate=40.0, lifetime=1.0, reporter_prod=40.0, reporter_decay=0.5, tmax=100, n_reps=30, seed=1):
    ratios = np.linspace(0.3, 1.5, 25)
    rng = np.random.default_rng(seed)
    fractions = []
    for ratio in ratios:
        rep_vals = []
        for _ in range(n_reps):
            T, X = gillespie_sini_sinr(ratio*lambda_R, lambda_R, complex_rate, lifetime, reporter_prod, reporter_decay, tmax, int(rng.integers(0, 2**32-1)))
            Z = X[:, 3]
            dt = np.diff(np.r_[T, tmax])
            rep_vals.append(np.sum(dt * (Z > 10)) / np.sum(dt))
        fractions.append(np.mean(rep_vals))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(ratios, fractions, 'o-', lw=2, ms=4)
    ax.set_xlabel('SinI / SinR production ratio')
    ax.set_ylabel('fraction of time reporter ON')
    ax.set_ylim(-0.05, 1.05)
    ax.set_title('State occupancy changes with production balance')
    plt.show()

interact(
    sweep_ratio,
    lambda_R=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='λR'),
    complex_rate=FloatSlider(value=40.0, min=0.1, max=100.0, step=1.0, description='complex'),
    lifetime=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='lifetime'),
    reporter_prod=FloatSlider(value=40.0, min=1, max=120, step=1, description='Z prod'),
    reporter_decay=FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05, description='Z decay'),
    tmax=IntSlider(value=100, min=20, max=300, step=10, description='Tmax'),
    n_reps=IntSlider(value=30, min=5, max=100, step=5, description='n_reps'),
    seed=IntSlider(value=1, min=0, max=1000, step=1, description='seed'),
);

interactive(children=(FloatSlider(value=1.0, description='λR', max=5.0, min=0.1), FloatSlider(value=40.0, desc…